# 项目架构学习文档

本笔记从整体视角讲解项目的架构设计，包括分层模式、数据流转、依赖注入、缓存与锁等内容。

## 内容概览

1. **整体架构** — 分层设计与模块划分
2. **数据流转** — 一个请求从进来到返回的完整路径
3. **Schema/DTO 模式** — 各层数据模型的职责与转换
4. **依赖注入** — FastAPI 的 Depends 机制
5. **缓存与 Redis** — CacheBackend 抽象与工厂模式
6. **Comic 模块** — 缓存 + 锁 + double-check 的综合运用
7. **测试架构** — 如何 mock 各层进行单元测试

## 一、整体架构

```
                    ┌─────────────┐
                    │   Client    │  浏览器 / API 客户端
                    └──────┬──────┘
                           │ HTTP 请求
                           ▼
                    ┌─────────────┐
                    │   FastAPI   │  路由分发 + 请求校验 + 依赖注入
                    └──────┬──────┘
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
        ┌──────────┐ ┌──────────┐ ┌──────────┐
        │  Router   │ │  Router   │ │  Router   │
        │  user.py  │ │ comic.py  │ │  (更多)   │  ← 接入层：接收请求，返回响应
        └─────┬────┘ └─────┬────┘ └──────────┘
              │            │
              ▼            ▼
        ┌──────────┐ ┌──────────┐
        │ Service   │ │ Service   │
        │  user.py  │ │ comic.py  │          ← 业务层：业务规则编排
        └─────┬────┘ └─────┬────┘
              │            │
              ▼            ▼
        ┌──────────┐ ┌──────────┐
        │   Repo    │ │  Cache    │
        │  user.py  │ │  + Lock   │          ← 数据层：ORM 操作 / 缓存操作
        └─────┬────┘ └─────┬────┘
              │            │
              ▼            ▼
        ┌──────────┐ ┌──────────┐
        │  MySQL    │ │  Redis/   │
        │ (SQLModel)│ │  Memory   │          ← 存储层：持久化 / 缓存
        └──────────┘ └──────────┘
```

### 分层职责

| 层 | 关注点 | 典型操作 | 不应该做的事 |
|---|---|---|---|
| **Router** | HTTP 协议 | 解析路径参数、校验请求体、注入依赖、设置状态码 | 不写业务逻辑，不操作数据库 |
| **Service** | 业务规则 | 检查重复、判断权限、编排多个 repo 调用 | 不写 SQL/ORM，不接触密码哈希 |
| **Repository** | 数据存取 | CRUD、密码哈希、ORM 查询 | 不关心业务规则（如"邮箱重复返回什么错误码"） |
| **Model** | 数据结构 | 定义字段、类型、约束 | 不包含行为逻辑 |

## 二、数据流转

以 `POST /user/register` 为例，展示数据在各层间的流转和类型变化：

```
客户端 JSON
  {"name": "张三", "password": "123", "email": "z@example.com"}
       │
       ▼  FastAPI 自动解析 + 校验
  UserCreateRequest(name="张三", password="123", email="z@example.com")    ← request.py
       │
       ▼  Router 提取字段
  基本类型参数: name="张三", password="123", email="z@example.com"
       │
       ▼  Service 调用 Repo
  User(name="张三", password="$2b$12$...", email="z@example.com")           ← tb_user.py (ORM)
       │
       ▼  Repo 内部: _to_public()
  UserPublic(id=1, name="张三", email="z@example.com")                      ← schemas.py (DTO)
       │
       ▼  Service 内部: _to_response()
  UserResponse(name="张三", email="z@example.com")                          ← response.py
       │
       ▼  包装为统一格式
  APIResponse(code=0, message="success", data={"name": "张三", ...})        ← response.py
       │
       ▼  FastAPI 自动序列化
  返回 JSON 给客户端
```

### 关键观察

1. **密码只在 Repo 层出现**：明文密码传给 repo，repo 内部哈希后存入数据库。返回时通过 `_to_public()` 剥离 password，Service 和上层永远不会看到密码。

2. **每一层有自己的数据模型**：Router 用 Request，Repo 用 ORM Model + DTO，Service 用 DTO + Response。各层通过转换函数衔接。

3. **错误处理沿层上传**：Repo 返回 `None` 或错误字符串，Service 翻译为 `APIResponse(code=X, message="...")`，Router 不关心错误细节。

## 三、Schema / DTO 模式

项目中有多种数据模型，每种服务于不同目的。理解它们的区别和协作是理解架构的关键。

### 1. ORM 模型 (`models/tb_user.py`)

```python
class User(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    password: str          # ← 包含敏感字段
    email: str = Field(unique=True)
```

- **用途**：直接映射数据库表
- **特点**：包含所有字段（含敏感字段如 password）
- **使用者**：只有 Repository 层

### 2. DTO / Schema (`models/schemas.py`)

```python
class UserPublic(BaseModel):
    id: int
    name: str
    email: str              # ← 没有 password！
```

- **用途**：Repository → Service 之间的数据传递
- **特点**：脱敏，不含敏感字段
- **使用者**：Repository（产出）→ Service（消费）

### 3. Request 模型 (`models/request.py`)

```python
class UserCreateRequest(BaseModel):
    name: str
    password: str          # ← 客户端传入的明文密码
    email: str
```

- **用途**：定义客户端应该传什么参数
- **特点**：FastAPI 自动校验类型、生成 OpenAPI 文档
- **使用者**：Router 接收，提取字段传给 Service

### 4. Response 模型 (`models/response.py`)

```python
class APIResponse(BaseModel):
    code: int = 0
    message: str = "success"
    data: Any = None

class UserResponse(BaseModel):
    name: str
    email: str              # ← 也不含 password
```

- **用途**：定义返回给客户端的数据格式
- **特点**：统一包装（APIResponse）+ 最小暴露原则
- **使用者**：Service 构造，Router 返回

### 数据模型转换链

```
                  Repo 内部                   Service 内部
User (ORM)  ──────────────→  UserPublic (DTO)  ──────────────→  UserResponse
含 password      _to_public()      不含 password     _to_response()    只有 name/email
                 剥离敏感字段                          进一步精简
```

### 为什么不直接复用一个模型？

用不同的模型看似啰嗦，但各有明确目的：
- **安全**：`User` 含密码，只允许 repo 层访问；`UserPublic` 脱敏后给 service；`UserResponse` 进一步精简给客户端
- **解耦**：改数据库字段不影响 API 接口；改 API 接口不影响数据库
- **明确**：看到函数签名就知道它接收什么、返回什么

## 四、依赖注入

FastAPI 的 `Depends()` 是本项目的核心机制之一，用于自动管理资源的创建和销毁。

### 4.1 数据库 Session 注入

```python
# core/database.py
def get_session() -> Generator[Session, Any, None]:
    with Session(engine) as session:
        yield session   # yield 前创建，yield 后清理

# routers/user.py
@router.post("/register")
def register(req: UserCreateRequest, session: Session = Depends(get_session)):
    #                   FastAPI 自动调用 get_session() ↑
    return user_service.register(session, ...)
```

流程：
1. 请求进来 → FastAPI 发现 `session: Session = Depends(get_session)`
2. 调用 `get_session()`，执行到 `yield`，得到一个 session
3. 把 session 传给路由函数
4. 路由函数处理完，`with Session(engine)` 的 `__exit__` 自动关闭 session

### 4.2 缓存注入

```python
# core/cache.py
_cache: CacheBackend | None = None

def get_cache() -> Generator[CacheBackend | None, Any, None]:
    global _cache
    if _cache is None:
        _cache = create_cache()   # 单例：只在第一次调用时创建
    yield _cache

# routers/comic.py
@router.get("/download/{album_id}")
def download(album_id: int, cache: CacheBackend = Depends(get_cache)):
    return jmcomic_service.download_and_merge_pdf(album_id, cache)
```

### 4.3 依赖注入的好处

| 好处 | 说明 |
|---|---|
| **自动管理生命周期** | session 用完自动关闭，不用担心忘记关闭 |
| **可测试** | 测试时用 `app.dependency_overrides[get_session] = fake_session` 替换 |
| **解耦** | 路由不关心 session 怎么创建的，只管用 |

## 五、缓存与 Redis

项目通过 `Protocol`（协议）定义缓存接口，支持 MemoryCache 和 RedisCache 两种实现。

### 5.1 架构图

```
              CacheBackend (Protocol)
              ├── get(key) → str | None
              ├── set(key, value, ex)
              ├── delete(key)
              └── exists(key) → bool
                     ▲         ▲
                     │         │
           ┌─────────┘         └─────────┐
           │                             │
    MemoryCache                     RedisCache
    ├─ dict[str, tuple]             ├─ 委托 redis_client
    ├─ RWLock 读写锁                └─ Redis 自带线程安全
    └─ 惰性过期                     └─ Redis 自带 TTL
           │                             │
           └───── create_cache() ────────┘
                    工厂函数
                  读取 CACHE_BACKEND
                  环境变量决定用哪个
```

### 5.2 Redis 配置 (`core/redis.py`)

```python
from redis import Redis, ConnectionPool

connection_pool = ConnectionPool(
    host=os.getenv("REDIS_HOST", "localhost"),
    port=int(os.getenv("REDIS_PORT", "6379")),
    password=os.getenv("REDIS_PASSWORD") or None,
    db=int(os.getenv("REDIS_DB", "0")),
    decode_responses=True,       # 返回 str 而不是 bytes
    max_connections=20,          # 连接池最大连接数
    socket_timeout=5,            # 读写超时
    socket_connect_timeout=5,    # 连接超时
)
redis_client = Redis(connection_pool=connection_pool)
```

### 5.3 工厂模式

```python
def create_cache() -> CacheBackend:
    backend = os.getenv("CACHE_BACKEND", "memory")
    if backend == "redis":
        return RedisCache()
    return MemoryCache()
```

通过环境变量 `CACHE_BACKEND` 切换后端，业务代码（service/router）完全不变。

| 环境 | CACHE_BACKEND | 实际使用 |
|---|---|---|
| 本地开发 | `memory`（默认） | MemoryCache，无需启动 Redis |
| 生产环境 | `redis` | RedisCache，使用 Redis 连接池 |
| 测试 | `memory` | MemoryCache，无需外部依赖 |

## 六、Comic 模块 — 缓存 + 锁的综合运用

Comic 模块展示了项目中最复杂的并发控制模式。

### 6.1 模块组成

```
routers/comic.py          services/comic.py
┌───────────────┐        ┌────────────────────────────┐
│ download()    │───────→│ download_and_merge_pdf()   │
│               │        │                            │
│ 注入 Cache    │        │ 使用 KeyLock + CacheBackend │
│ 通过 Depends  │        │ Double-Check Locking 模式   │
└───────────────┘        └────────────────────────────┘
```

### 6.2 Router

```python
# routers/comic.py
router = APIRouter(prefix="/jm", tags=["jm"])

@router.get("/download/{album_id}")
def download(album_id: int, cache: CacheBackend = Depends(get_cache)):
    return jmcomic_service.download_and_merge_pdf(album_id, cache)
```

注意这个路由注入的是 `get_cache` 而不是 `get_session`，因为 comic 功能不需要数据库。

### 6.3 Service — Double-Check Locking

```python
# services/comic.py
_album_lock = KeyLock()   # 模块级单例

def download_and_merge_pdf(album_id: int, cache: CacheBackend) -> PdfFileResponse:
    cache_key = f"jmcomic:{album_id}"

    # ① 快速路径：缓存命中 + 文件存在 → 不加锁直接返回
    cached_path = cache.get(cache_key)
    if cached_path is not None:
        pdf_path = Path(cached_path)
        if pdf_path.exists():
            return PdfFileResponse(pdf_path, filename=pdf_path.name)
        cache.delete(cache_key)   # 文件已不存在，清理脏缓存

    # ② 对同一 album_id 加锁（不同 album_id 互不影响）
    with _album_lock.acquire(album_id):
        # ③ double-check：拿到锁后再查一次缓存
        cached_path = cache.get(cache_key)
        if cached_path is not None:
            pdf_path = Path(cached_path)
            if pdf_path.exists():
                return PdfFileResponse(pdf_path, filename=pdf_path.name)
            cache.delete(cache_key)

        # ④ 真正执行下载 + 写入缓存
        download_album(album_id, option)
        pdf_path = DOWNLOAD_DIR / f"{album_id}.pdf"
        cache.set(cache_key, str(pdf_path))

    return PdfFileResponse(pdf_path, filename=pdf_path.name)
```

### 6.4 并发场景分析

```
假设 3 个请求同时请求 album_id=123：

Req-A ─┐
Req-B ─┤ 快速路径都未命中
Req-C ─┘
        │
        ├─ 只有 Req-A 拿到锁，B、C 排队
        │
Req-A: 下载完成 → cache.set() → 释放锁
        │
Req-B: 拿到锁 → double-check 命中 → 直接返回（不重复下载）
        │
Req-C: 拿到锁 → double-check 命中 → 直接返回（不重复下载）
```

### 6.5 同时请求不同 album_id

```
Req-1 (album_id=100) ──→ KeyLock[100] ──→ 下载 ──→ 返回
                                              ↕ 并发，互不影响
Req-2 (album_id=200) ──→ KeyLock[200] ──→ 下载 ──→ 返回
```

`KeyLock` 保证不同 album_id 的锁互不干扰，不会出现下载 ID=100 时阻塞 ID=200 的情况。

## 七、测试架构

项目的测试利用了分层架构的解耦特性，通过 mock 替换各层依赖。

### 7.1 测试数据库替换 (`conftest.py`)

```python
@pytest.fixture(name="client")
def client_fixture(engine):
    # 用 SQLite 内存数据库替代 MySQL
    def override_get_session():
        with Session(engine) as session:
            yield session

    # 替换 create_db_and_tables 为空操作，避免连接真实 MySQL
    main_module.create_db_and_tables = lambda: None

    # 通过 dependency_overrides 注入测试 session
    app.dependency_overrides[get_session] = override_get_session
    with TestClient(app) as client:
        yield client
    app.dependency_overrides.clear()
```

关键点：
- **SQLite 内存数据库**替代 MySQL，无需启动真实数据库
- **`dependency_overrides`** 替换 `get_session`，所有路由自动使用测试 session
- **monkey-patch** `create_db_and_tables` 避免启动时连接 MySQL

### 7.2 测试层次

```
┌─────────────────────────────────────────┐
│  test_routers_user.py                    │  集成测试
│  使用 TestClient + SQLite 内存数据库      │  测试完整请求流程
│  覆盖：Router → Service → Repo → DB      │
└─────────────────────────────────────────┘

┌─────────────────────────────────────────┐
│  test_services_user.py                   │  单元测试
│  mock repo 层方法                         │  测试业务逻辑
│  覆盖：Service 层的业务判断               │  不依赖数据库
└─────────────────────────────────────────┘
```

### 7.3 Service 单元测试示例

因为 Service 层只调用 Repo 方法，所以可以轻松 mock：

```python
from unittest.mock import MagicMock

def test_register_email_duplicate():
    session = MagicMock()
    # mock repo 层返回一个已存在的用户
    user_repo.get_by_email = MagicMock(return_value=UserPublic(id=1, ...))

    result = user_service.register(session, "张三", "123", "z@example.com")

    assert result.code == 1
    assert result.message == "email already registered"
    user_repo.create.assert_not_called()   # 不应该调用 create
```

分层架构让测试变得简单：
- 测试 Router → 用 TestClient + 内存数据库
- 测试 Service → mock Repo 方法
- 测试 Repo → 用真实内存数据库

---

## 总结：架构设计原则

| 原则 | 在项目中的体现 |
|---|---|
| **单一职责** | Router 只管 HTTP，Service 只管业务，Repo 只管数据 |
| **面向接口编程** | `CacheBackend` Protocol，Service 不依赖具体缓存实现 |
| **最小暴露** | `UserPublic` 脱敏，`UserResponse` 进一步精简 |
| **依赖注入** | `Depends(get_session)` / `Depends(get_cache)` 解耦资源管理 |
| **工厂模式** | `create_cache()` 根据环境变量选择缓存后端 |
| **锁粒度最小化** | `KeyLock` 按 album_id 分锁，`RWLock` 读写分离 |
| **快速路径优先** | 先不加锁查缓存，命中就直接返回 |
| **Double-Check** | 拿到锁后再确认，避免并发重复工作 |